In [ ]:
# Device Agnostic code
import torch

device = "cuda" if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [72]:
# Loading `Fashion_MNIST` Dataset
from torchvision import datasets, transforms

# Define transformations
transform = transforms.Compose([transforms.ToTensor()])

# Download and load the training data
train_data = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)

# Download and load the test data
test_data = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)

100%|██████████| 26.4M/26.4M [00:07<00:00, 3.49MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 161kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 1.69MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 7.81MB/s]


In [ ]:
# Custom Dataset
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, dataset_obj) -> None:
        self.dataset = dataset_obj
        super().__init__()
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, index):
        return self.dataset[index][0].ravel(), self.dataset[index][1]

In [123]:
# Creating DataLoaders
train_dataset = CustomDataset(dataset_obj=train_data)
test_dataset = CustomDataset(dataset_obj=test_data)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=5,
    shuffle=True,
    pin_memory=True,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=5,
    shuffle=True,
    pin_memory=True,
    drop_last=False,
)

In [ ]:
# Custom Model
from torch import nn

class CustomModel(nn.Module):
    def __init__(self, features_count: int) -> None:
        super().__init__()

        # Model Architecture
        self.model = nn.Sequential(
            nn.Linear(in_features=features_count, out_features=50),
            nn.ReLU(),
            nn.Linear(in_features=50, out_features=25),
            nn.ReLU(),
            nn.Linear(in_features=25, out_features=10),
        )
    
    def forward(self, features):
        return self.model(features)

In [125]:
# Optimization Parameters
learning_rate = 0.001
epochs = 10

In [126]:
# Loss function and Optimizer
model = CustomModel(features_count=28*28).to(device=device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.999)
)

In [128]:
# Training pipeline
for i in range(epochs):
    total_loss = 0

    for features, labels in train_loader:
        # Shifting features and labels to GPU
        features = features.to(device)
        labels = labels.to(device)

        # Forward Propogation
        y_pred = model(features)

        # Calculate the loss
        loss = criterion(y_pred, labels)

        # Backword Propogation
        optimizer.zero_grad()
        loss.backward()

        # Update the parameters
        optimizer.step()
    
        # Loss
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Loss at epoch {i + 1} = {avg_loss}")

Loss at epoch 1 = 0.45406166037002244
Loss at epoch 2 = 0.3853649497360117
Loss at epoch 3 = 0.3561970864205741
Loss at epoch 4 = 0.3388256155972221
Loss at epoch 5 = 0.3253771084825122
Loss at epoch 6 = 0.3161423760472039
Loss at epoch 7 = 0.3064624700706073
Loss at epoch 8 = 0.29899429757562324
Loss at epoch 9 = 0.29470046444222575
Loss at epoch 10 = 0.2873440233464761


In [129]:
# Evaluation Mode
model.eval()

CustomModel(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=50, bias=True)
    (1): ReLU()
    (2): Linear(in_features=50, out_features=25, bias=True)
    (3): ReLU()
    (4): Linear(in_features=25, out_features=10, bias=True)
  )
)

In [133]:
# Evaluation pipeline
with torch.no_grad():
    train_correct = 0
    test_correct = 0

    for features, labels in train_loader:
        # Shifting features and labels to GPU
        features = features.to(device)
        labels = labels.to(device)
        
        # Model prediction
        y_pred = model(features).argmax(dim=1)

        # Calculating accuracy
        train_correct += (labels == y_pred).sum()

    for features, labels in test_loader:
        # Shifting features and labels to GPU
        features = features.to(device)
        labels = labels.to(device)
        
        # Model prediction
        y_pred = model(features).argmax(dim=1)

        # Calculating accuracy
        test_correct += (labels == y_pred).sum()
    
    print("Training Accuracy =", train_correct / len(train_dataset))
    print("Testing Accuracy =", test_correct / len(test_dataset))

Training Accuracy = tensor(0.9000, device='cuda:0')
Testing Accuracy = tensor(0.8732, device='cuda:0')
